In [72]:
import os
import json
import re
import pandas as pd

# object oriented approach to working with paths
from pathlib import Path

import duckdb

In [2]:
# ensuring that os.chdir is idempotent and that we are in the project root directory,
# not inside the notebooks directory
if 'notebooks' not in os.listdir(Path.cwd()):
    print("Still inside notebooks directory, changing to project root directory.")
    os.chdir(Path.cwd().parent)
    print("Current working directory after change: ", Path.cwd())
else:
    print(f"Already in parent directory (current working directory: {Path.cwd()})")


# load config
from src.io.load_config import load_config
sw_config = load_config()['space_weather']
omni_config = load_config()['omni']

Still inside notebooks directory, changing to project root directory.
Current working directory after change:  d:\data-sci-projects\space-weather-project-scrub


In [12]:
omni_config

{'hapi': {'base_url': 'https://cdaweb.gsfc.nasa.gov/hapi',
  'supported_version': '2.0',
  'dataset_id': 'OMNI_HRO2_1MIN',
  'chunk_days': 10,
  'timeout_s': 120,
  'sleep_s': 5,
  'raw_output_dir': 'data/01-raw/omni',
  'CLI_UTC_FMT': '%Y-%m-%d %H:%M:%S',
  'HAPI_UTC_FMT': '%Y-%m-%dT%H:%M:%SZ'}}

# Grab ingestion aratifacts

In [123]:
raw_data_dir = Path(omni_config['hapi']['raw_output_dir']) / 'OMNI_HRO2_1MIN'
run_dirs = [Path(raw_data_dir)/ run_id for run_id in os.listdir(raw_data_dir)]
json_glob_run_dirs = [(Path(run_dir) / 'chunk*.json').as_posix() for run_dir in run_dirs]
manifest_run_dirs = [(Path(run_dir) / '_manifest.json').as_posix() for run_dir in run_dirs]
hapi_info_dirs = [(Path(run_dir) / 'hapi_info.json').as_posix() for run_dir in run_dirs]


In [124]:
json_glob_run_dirs

['data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260720T035848Z/chunk*.json',
 'data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260720T035942Z/chunk*.json',
 'data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260720T040507Z/chunk*.json',
 'data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260801T021300Z/chunk*.json',
 'data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260801T021613Z/chunk*.json']

In [101]:
pd.set_option('display.max_colwidth', None)

','.join(
    duckdb.execute(f"""
        SELECT
            run_id,
            unnest(parameters)->>'name' AS name
        FROM read_json({hapi_info_dirs})
    """).fetch_df().name.unique()
)



'Time,IMF,PLS,IMF_PTS,PLS_PTS,percent_interp,Timeshift,RMS_Timeshift,RMS_phase,Time_btwn_obs,F,BX_GSE,BY_GSE,BZ_GSE,BY_GSM,BZ_GSM,RMS_SD_B,RMS_SD_fld_vec,flow_speed,Vx,Vy,Vz,proton_density,T,NaNp_Ratio,Pressure,E,Beta,Mach_num,Mgs_mach_num,x,y,z,BSN_x,BSN_y,BSN_z,AE_INDEX,AL_INDEX,AU_INDEX,SYM_D,SYM_H,ASY_D,ASY_H'

In [110]:
pd.set_option('display.max_colwidth', None)
duckdb.execute(f"""
    SELECT
        run_id,
        request->>'effective_start_utc' AS effective_start_utc,
        request->>'effective_end_utc' AS effective_end_utc,
        request->>'parameters' AS parameters,
        request->>'time_range_overlap_status' AS time_range_overlap_status,
    FROM read_json({manifest_run_dirs})
    ORDER BY effective_start_utc, effective_end_utc
""").fetch_df()

,run_id,effective_start_utc,effective_end_utc,parameters,time_range_overlap_status
0,20260801T021300Z,2025-01-01 00:00:00,2025-02-01 00:00:00,"[""Time"",""IMF"",""PLS"",""IMF_PTS"",""PLS_PTS"",""percent_interp"",""Timeshift"",""RMS_Timeshift"",""RMS_phase"",""Time_btwn_obs"",""F"",""BX_GSE"",""BY_GSE"",""BZ_GSE"",""BY_GSM"",""BZ_GSM"",""RMS_SD_B"",""RMS_SD_fld_vec"",""flow_speed"",""Vx"",""Vy"",""Vz"",""proton_density"",""T"",""NaNp_Ratio"",""Pressure"",""E"",""Beta"",""Mach_num"",""Mgs_mach_num"",""x"",""y"",""z"",""BSN_x"",""BSN_y"",""BSN_z"",""AE_INDEX"",""AL_INDEX"",""AU_INDEX"",""SYM_D"",""SYM_H"",""ASY_D"",""ASY_H""]",subset
1,20260801T021613Z,2025-02-01 00:00:00,2025-05-01 00:00:00,"[""Time"",""IMF"",""PLS"",""IMF_PTS"",""PLS_PTS"",""percent_interp"",""Timeshift"",""RMS_Timeshift"",""RMS_phase"",""Time_btwn_obs"",""F"",""BX_GSE"",""BY_GSE"",""BZ_GSE"",""BY_GSM"",""BZ_GSM"",""RMS_SD_B"",""RMS_SD_fld_vec"",""flow_speed"",""Vx"",""Vy"",""Vz"",""proton_density"",""T"",""NaNp_Ratio"",""Pressure"",""E"",""Beta"",""Mach_num"",""Mgs_mach_num"",""x"",""y"",""z"",""BSN_x"",""BSN_y"",""BSN_z"",""AE_INDEX"",""AL_INDEX"",""AU_INDEX"",""SYM_D"",""SYM_H"",""ASY_D"",""ASY_H""]",subset
2,20260720T040507Z,2025-11-01 00:00:00,2026-01-01 00:00:00,"[""F"",""BX_GSE"",""BY_GSM"",""BZ_GSM"",""flow_speed"",""proton_density"",""Pressure""]",subset
3,20260720T035848Z,2026-01-01 00:00:00,2026-01-02 00:00:00,"[""F"",""BX_GSE"",""BY_GSM"",""BZ_GSM"",""flow_speed"",""proton_density"",""Pressure""]",subset
4,20260720T035942Z,2026-01-01 00:00:00,2026-03-02 00:00:00,"[""F"",""BX_GSE"",""BY_GSM"",""BZ_GSM"",""flow_speed"",""proton_density"",""Pressure""]",subset


# 1. Run audit

In [138]:
manifest_run_dirs

['data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260720T035848Z/_manifest.json',
 'data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260720T035942Z/_manifest.json',
 'data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260720T040507Z/_manifest.json',
 'data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260801T021300Z/_manifest.json',
 'data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260801T021613Z/_manifest.json']

In [ ]:
pd.set_option('display.max_colwidth', None)

# IMPORTANT NOTE (so that code is self-explaining)
# run id is automatically extracted from the filename.
# But best if we can make this syntactically explicit
# to minimize any surprises.

duckdb.execute(f"""
    SELECT
        run_id,
        source->>'dataset_id' AS dataset_id,
        request->>'requested_start_utc' AS requested_start_utc,
        request->>'requested_end_utc' AS requested_end_utc,
        request->>'effective_start_utc' AS effective_start_utc,
        request->>'effective_end_utc' AS effective_end_utc,
        request->>'time_range_overlap_status' AS time_range_overlap_status,
        summary->>'total_rows' AS total_rows,
        summary->>'empty_chunk_count' AS empty_chunk_count
    FROM read_json({manifest_run_dirs})
    WHERE run->>'status' = 'SUCCESS'
    ORDER BY effective_start_utc, effective_end_utc
""").fetch_df()

,run_id,dataset_id,requested_start_utc,requested_end_utc,effective_start_utc,effective_end_utc,time_range_overlap_status,total_rows,empty_chunk_count
0,20260801T021300Z,OMNI_HRO2_1MIN,2025-01-01 00:00:00,2025-02-01 00:00:00,2025-01-01 00:00:00,2025-02-01 00:00:00,subset,44640,0
1,20260801T021613Z,OMNI_HRO2_1MIN,2025-02-01 00:00:00,2025-05-01 00:00:00,2025-02-01 00:00:00,2025-05-01 00:00:00,subset,128160,0
2,20260720T040507Z,OMNI_HRO2_1MIN,2025-11-01 00:00:00,2026-01-01 00:00:00,2025-11-01 00:00:00,2026-01-01 00:00:00,subset,87840,0
3,20260720T035848Z,OMNI_HRO2_1MIN,2026-01-01 00:00:00,2026-01-02 00:00:00,2026-01-01 00:00:00,2026-01-02 00:00:00,subset,1440,0
4,20260720T035942Z,OMNI_HRO2_1MIN,2026-01-01 00:00:00,2026-03-02 00:00:00,2026-01-01 00:00:00,2026-03-02 00:00:00,subset,86400,0


# 2. Long observation audit

In [125]:
json_glob_run_dirs

['data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260720T035848Z/chunk*.json',
 'data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260720T035942Z/chunk*.json',
 'data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260720T040507Z/chunk*.json',
 'data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260801T021300Z/chunk*.json',
 'data/01-raw/omni/OMNI_HRO2_1MIN/run_id=20260801T021613Z/chunk*.json']

[Duckdb tips on unnesting](https://duckdb.org/docs/lts/sql/query_syntax/unnest)

Unnesting a list generates rows
- `SELECT unnest([1, 2, 3]);` generates 3 rows `[1,2,3]`

Unnesting a struct generates columns
- `SELECT unnest({'a': 42, 'b': 84});` generates 2 columns `(a,b)`

Unnest a list of structs into a table (basically similar to `pd.DataFrame(list of dicts)`)

```
> SELECT unnest([{'a': 42, 'b': 84}, {'a': 100, 'b': NULL}], recursive := true);
>
	a	b
0	42	84
1	100	<NA>
```


Let's focus on *one* run first.

How do we unnest each multivariate observation into rows.....

In [177]:
pd.set_option('display.max_colwidth', None)


# IMPORTANT NOTE (so that code is self-explaining)
# run id is automatically extracted from the filename.
# But best if we can make this syntactically explicit
# to minimize any surprises.

# filter these rows by doing an INNER JOIN with the above to
# only take chunks from successful rows.

duckdb.execute(f"""
    WITH
        unnested_data AS (
            SELECT
                unnest(data)
            FROM read_json({json_glob_run_dirs[0:1]}, filename=true)
        ),
        unnested_param AS (
            SELECT
                unnest(parameters, recursive := true)
            FROM read_json({json_glob_run_dirs[0:1]}, filename=true)
        )

    SELECT *
    FROM unnested_data 
""").fetch_df()

,"unnest(""data"")"
0,"[""2026-01-01T00:00:00.000Z"", 9.85, -1.39, -9.71, -0.26, 99999.9, 999.99, 99.99]"
1,"[""2026-01-01T00:01:00.000Z"", 9.77, 0.17, -9.3, 1.56, 99999.9, 999.99, 99.99]"
2,"[""2026-01-01T00:02:00.000Z"", 9.65, -1.95, -8.8, 1.69, 99999.9, 999.99, 99.99]"
3,"[""2026-01-01T00:03:00.000Z"", 9.63, -1.99, -8.94, 1.71, 99999.9, 999.99, 99.99]"
4,"[""2026-01-01T00:04:00.000Z"", 9.73, -2.08, -9.39, 1.22, 99999.9, 999.99, 99.99]"
...,...
1435,"[""2026-01-01T23:55:00.000Z"", 6.59, -2.68, -2.52, 5.38, 560.6, 5.51, 3.46]"
1436,"[""2026-01-01T23:56:00.000Z"", 6.7, -3.18, -1.74, 5.59, 560.6, 5.51, 3.46]"
1437,"[""2026-01-01T23:57:00.000Z"", 6.66, -2.02, -3.43, 5.26, 553.4, 5.43, 3.33]"
1438,"[""2026-01-01T23:58:00.000Z"", 6.68, -1.74, -4.01, 5.03, 553.4, 5.43, 3.33]"


....whilst keeping track of the list indices in `/data.json['parameters']` so that each number can be traced back to the correct parameter name & fill & type

In [178]:
pd.set_option('display.max_colwidth', None)


# IMPORTANT NOTE (so that code is self-explaining)
# run id is automatically extracted from the filename.
# But best if we can make this syntactically explicit
# to minimize any surprises.

# filter these rows by doing an INNER JOIN with the above to
# only take chunks from successful rows.

duckdb.execute(f"""
    WITH
        unnested_data AS (
            SELECT
                unnest(data)
            FROM read_json({json_glob_run_dirs[0:1]}, filename=true)
        ),
        unnested_param AS (
            SELECT
                unnest(parameters, recursive := true)
            FROM read_json({json_glob_run_dirs[0:1]}, filename=true)
        )

    SELECT *
    FROM unnested_param 
""").fetch_df()

,fill,length,name,type,units,description
0,None,24,Time,isotime,UTC,None
1,9999.99,<NA>,F,double,nT,"Magnitude of avg. field vector (nT) (last currently-available OMNI B-field data Apr 12, 2026)"
2,9999.99,<NA>,BX_GSE,double,nT,"Bx (nT), GSE"
3,9999.99,<NA>,BY_GSM,double,nT,"By (nT), GSM, determined from post-shift GSE components"
4,9999.99,<NA>,BZ_GSM,double,nT,"Bz (nT), GSM, determined from post-shift GSE components"
5,99999.9,<NA>,flow_speed,double,km/s,"Flow Speed (km/s), GSE"
6,999.99,<NA>,proton_density,double,n/cc,"Proton density (n/cc) (last currently-available OMNI plasma data Apr 12, 2026)"
7,99.99,<NA>,Pressure,double,nPa,Flow pressure (nPa)


solution for now: use `UNNEST(ARRAY) WITH ORDINALITY AS t(element, index)`
For example,
```
> SELECT element, index
FROM UNNEST(ARRAY['apple', 'banana', 'cherry']) 
WITH ORDINALITY AS t(element, index);

> 
element | index
apple | 1
banana | 2 
cherry | 3
```

In [176]:
duckdb.execute(f"""
    WITH unnested_data AS (
        SELECT
            run_id,
            unnest(data) AS row_array
        FROM read_json({json_glob_run_dirs[0:1]}, filename=true)
    )
    SELECT 
        run_id,
        row_array[1] AS timestamp,
        val AS value,
        param_idx
    FROM unnested_data,
    UNNEST(row_array[2:]) WITH ORDINALITY AS t(val, param_idx)
    ORDER BY run_id, timestamp, param_idx
""").fetch_df()

,run_id,timestamp,value,param_idx
0,20260720T035848Z,"""2026-01-01T00:00:00.000Z""",9.85,1
1,20260720T035848Z,"""2026-01-01T00:00:00.000Z""",-1.39,2
2,20260720T035848Z,"""2026-01-01T00:00:00.000Z""",-9.71,3
3,20260720T035848Z,"""2026-01-01T00:00:00.000Z""",-0.26,4
4,20260720T035848Z,"""2026-01-01T00:00:00.000Z""",99999.9,5
...,...,...,...,...
10075,20260720T035848Z,"""2026-01-01T23:59:00.000Z""",-4.52,3
10076,20260720T035848Z,"""2026-01-01T23:59:00.000Z""",4.76,4
10077,20260720T035848Z,"""2026-01-01T23:59:00.000Z""",99999.9,5
10078,20260720T035848Z,"""2026-01-01T23:59:00.000Z""",999.99,6


Then we join with `unnested_param` as defined above (create a CTE expression perhaps), and use regex to extract chunk filenames from `filename`.

In [ ]:
long_observations = duckdb.execute(f"""

    -- 1. Retrieve chunks from successful runs
    WITH successful_runs AS (
        SELECT
            run->>'run_id' AS run_id,
            source->>'dataset_id' AS dataset_id
        FROM read_json(
            {manifest_run_dirs},
            filename = true
        )
        WHERE run->>'status' = 'SUCCESS'
    ),

    successful_chunks AS (
        SELECT
            successful_runs.dataset_id,
            chunks.run_id,
            chunks.filename,
            chunks.data, -- still a list of lists at this point
            chunks.parameters
        FROM read_json(
            {json_glob_run_dirs},
            filename = true,
            hive_partitioning = true
        ) AS chunks
        INNER JOIN successful_runs
            ON chunks.run_id = successful_runs.run_id
    ),

    observation_rows AS (
        SELECT
            dataset_id,
            run_id,
            filename,
            source_row_number,
            row_array
        FROM successful_chunks
        CROSS JOIN UNNEST(data)
            WITH ORDINALITY AS observations(
                row_array,
                source_row_number
            )
    ),

    observation_values AS (
        SELECT
            dataset_id,
            run_id,
            filename,
            source_row_number,
            CAST(
                json_extract_string(row_array[1], '$')
                AS TIMESTAMPTZ
            ) AT TIME ZONE 'UTC' AS observation_time_utc,
            parameter_index,
            CAST(raw_value AS DOUBLE) AS raw_value
        FROM observation_rows
        CROSS JOIN UNNEST(row_array[2:])
            WITH ORDINALITY AS values_long(
                raw_value,
                parameter_index
            )
    ),

    parameter_definitions AS (
        SELECT
            dataset_id,
            run_id,
            filename,
            parameter_index,
            parameter.name AS parameter_name,
            CAST(parameter.fill AS DOUBLE) AS source_fill_value,
            parameter.units AS units,
            parameter.type AS parameter_type
        FROM successful_chunks
        CROSS JOIN UNNEST(parameters[2:])
            WITH ORDINALITY AS definitions(
                parameter,
                parameter_index
            )
    )

    SELECT
        values.dataset_id,
        values.run_id,
        regexp_extract(values.filename, '[^/]+$') AS chunk_file,
        values.source_row_number,
        values.observation_time_utc,
        parameters.parameter_name,
        values.raw_value,
        parameters.source_fill_value,
        parameters.units,
        parameters.parameter_type,
        (
            parameters.source_fill_value IS NOT NULL
            AND values.raw_value = parameters.source_fill_value
        ) AS is_source_fill
    FROM observation_values AS values
    INNER JOIN parameter_definitions AS parameters
        USING (
            dataset_id,
            run_id,
            filename,
            parameter_index
        )
    ORDER BY
        values.run_id,
        chunk_file,
        values.source_row_number,
        values.parameter_index
""").fetch_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [180]:
long_observations

,dataset_id,run_id,chunk_file,source_row_number,observation_time_utc,parameter_name,raw_value,source_fill_value,units,parameter_type,is_source_fill
0,OMNI_HRO2_1MIN,20260720T035848Z,data\01-raw\omni\OMNI_HRO2_1MIN\run_id=20260720T035848Z\chunk_20260101T000000Z__20260102T000000Z.json,1,2026-01-01 00:00:00,F,9.85,9999.99,nT,double,False
1,OMNI_HRO2_1MIN,20260720T035848Z,data\01-raw\omni\OMNI_HRO2_1MIN\run_id=20260720T035848Z\chunk_20260101T000000Z__20260102T000000Z.json,1,2026-01-01 00:00:00,BX_GSE,-1.39,9999.99,nT,double,False
2,OMNI_HRO2_1MIN,20260720T035848Z,data\01-raw\omni\OMNI_HRO2_1MIN\run_id=20260720T035848Z\chunk_20260101T000000Z__20260102T000000Z.json,1,2026-01-01 00:00:00,BY_GSM,-9.71,9999.99,nT,double,False
3,OMNI_HRO2_1MIN,20260720T035848Z,data\01-raw\omni\OMNI_HRO2_1MIN\run_id=20260720T035848Z\chunk_20260101T000000Z__20260102T000000Z.json,1,2026-01-01 00:00:00,BZ_GSM,-0.26,9999.99,nT,double,False
4,OMNI_HRO2_1MIN,20260720T035848Z,data\01-raw\omni\OMNI_HRO2_1MIN\run_id=20260720T035848Z\chunk_20260101T000000Z__20260102T000000Z.json,1,2026-01-01 00:00:00,flow_speed,99999.90,99999.90,km/s,double,True
...,...,...,...,...,...,...,...,...,...,...,...
8487355,OMNI_HRO2_1MIN,20260801T021613Z,data\01-raw\omni\OMNI_HRO2_1MIN\run_id=20260801T021613Z\chunk_20250422T000000Z__20250501T000000Z.json,12960,2025-04-30 23:59:00,AU_INDEX,81.00,99999.00,nT,integer,False
8487356,OMNI_HRO2_1MIN,20260801T021613Z,data\01-raw\omni\OMNI_HRO2_1MIN\run_id=20260801T021613Z\chunk_20250422T000000Z__20250501T000000Z.json,12960,2025-04-30 23:59:00,SYM_D,-2.00,99999.00,nT,integer,False
8487357,OMNI_HRO2_1MIN,20260801T021613Z,data\01-raw\omni\OMNI_HRO2_1MIN\run_id=20260801T021613Z\chunk_20250422T000000Z__20250501T000000Z.json,12960,2025-04-30 23:59:00,SYM_H,22.00,99999.00,nT,integer,False
8487358,OMNI_HRO2_1MIN,20260801T021613Z,data\01-raw\omni\OMNI_HRO2_1MIN\run_id=20260801T021613Z\chunk_20250422T000000Z__20250501T000000Z.json,12960,2025-04-30 23:59:00,ASY_D,20.00,99999.00,nT,integer,False


# Validating and modularizing long observation audit
To get a feel of how robust and correct the above query is:
- Break up the long query above into modular functions
- Selectively choose runs to test the output.
  - I choose two small runs (~1400 rows) with both `subset` and `partial` time overlap status

In [ ]:
# unify all ingestion-related directories in a class
# every instantiation -> grabs latest directories

class IngestionDirs:

    def __init__(self, raw_data_dir):
        self.run_dirs = [Path(raw_data_dir)/ run_id for run_id in os.listdir(raw_data_dir)]
        self.api_info_dirs = [(Path(run_dir) / 'hapi_info.json').as_posix() for run_dir in self.run_dirs]
        self.json_glob_run_dirs = [(Path(run_dir) / 'chunk*.json').as_posix() for run_dir in self.run_dirs]
        self.manifest_run_dirs = [(Path(run_dir) / '_manifest.json').as_posix() for run_dir in self.run_dirs]

    


In [200]:
def _as_duckdb_path_list(paths: list[str], argument_name: str) -> str:
    """Return normalized paths as a DuckDB list literal."""
    if not paths:
        raise ValueError(f"{argument_name} must not be empty")

    normalized_paths = [Path(path).as_posix() for path in paths]
    return repr(normalized_paths)


def _build_successful_runs_select_sql(manifest_paths: list[str]) -> str:
    """Build the successful-run manifest relation."""
    manifest_paths_sql = _as_duckdb_path_list(
        manifest_paths,
        "manifest_paths",
    )

    return f"""
        SELECT
            run->>'run_id' AS run_id,
            source->>'dataset_id' AS dataset_id
        FROM read_json(
            {manifest_paths_sql},
            filename = true
        )
        WHERE run->>'status' = 'SUCCESS'
    """.strip()


def _build_successful_chunks_select_sql(chunk_paths: list[str]) -> str:
    """Build chunks belonging to the successful-runs relation."""
    chunk_paths_sql = _as_duckdb_path_list(
        chunk_paths,
        "chunk_paths",
    )

    return f"""
        SELECT
            successful_runs.dataset_id,
            chunks.run_id,
            chunks.filename,
            chunks.data,
            chunks.parameters
        FROM read_json(
            {chunk_paths_sql},
            filename = true,
            hive_partitioning = true
        ) AS chunks
        INNER JOIN successful_runs
            ON chunks.run_id = successful_runs.run_id
    """.strip()


def _build_observation_rows_select_sql() -> str:
    """Build one row per source observation array."""
    return """
        SELECT
            dataset_id,
            run_id,
            filename,
            source_row_number,
            row_array
        FROM successful_chunks
        CROSS JOIN UNNEST(data)
            WITH ORDINALITY AS observations(
                row_array,
                source_row_number
            )
    """.strip()


def _build_observation_values_select_sql() -> str:
    """Build one value row per non-time observation parameter."""
    return """
        SELECT
            dataset_id,
            run_id,
            filename,
            source_row_number,
            CAST(
                json_extract_string(row_array[1], '$')
                AS TIMESTAMPTZ
            ) AT TIME ZONE 'UTC' AS observation_time_utc,
            parameter_index,
            CAST(raw_value AS DOUBLE) AS raw_value
        FROM observation_rows
        CROSS JOIN UNNEST(row_array[2:])
            WITH ORDINALITY AS values_long(
                raw_value,
                parameter_index
            )
    """.strip()


def _build_parameter_definitions_select_sql() -> str:
    """Build positional metadata for each non-time parameter."""
    return """
        SELECT
            dataset_id,
            run_id,
            filename,
            parameter_index,
            parameter.name AS parameter_name,
            CAST(parameter.fill AS DOUBLE) AS source_fill_value,
            parameter.units AS units,
            parameter.type AS parameter_type
        FROM successful_chunks
        CROSS JOIN UNNEST(parameters[2:])
            WITH ORDINALITY AS definitions(
                parameter,
                parameter_index
            )
    """.strip()


def build_long_observation_select_sql(
    manifest_paths: list[str],
    chunk_paths: list[str],
) -> str:
    """Build the complete long-observation audit query."""
    successful_runs_sql = _build_successful_runs_select_sql(
        manifest_paths
    )
    successful_chunks_sql = _build_successful_chunks_select_sql(
        chunk_paths
    )
    observation_rows_sql = _build_observation_rows_select_sql()
    observation_values_sql = _build_observation_values_select_sql()
    parameter_definitions_sql = (
        _build_parameter_definitions_select_sql()
    )

    return f"""
        WITH successful_runs AS (
            {successful_runs_sql}
        ),
        successful_chunks AS (
            {successful_chunks_sql}
        ),
        observation_rows AS (
            {observation_rows_sql}
        ),
        observation_values AS (
            {observation_values_sql}
        ),
        parameter_definitions AS (
            {parameter_definitions_sql}
        )
        SELECT
            values.dataset_id,
            values.run_id,
            regexp_extract(values.filename, '[^/]+$') AS chunk_file,
            values.source_row_number,
            values.observation_time_utc,
            parameters.parameter_name,
            values.raw_value,
            parameters.source_fill_value,
            parameters.units,
            parameters.parameter_type,
            (
                parameters.source_fill_value IS NOT NULL
                AND values.raw_value = parameters.source_fill_value
            ) AS is_source_fill
        FROM observation_values AS values
        INNER JOIN parameter_definitions AS parameters
            USING (dataset_id, run_id, filename, parameter_index)
        ORDER BY
            values.run_id,
            chunk_file,
            values.source_row_number,
            values.parameter_index
    """.strip()

In [201]:
dirs = IngestionDirs("temp/omni-ingest/")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE TEMP VIEW successful_runs AS
{_build_successful_runs_select_sql(dirs.manifest_run_dirs)}
""")

con.execute(f"""
CREATE OR REPLACE TEMP VIEW successful_chunks AS
{_build_successful_chunks_select_sql(dirs.json_glob_run_dirs)}
""")

con.execute(f"""
CREATE OR REPLACE TEMP VIEW observation_rows AS
{_build_observation_rows_select_sql()}
""")

con.execute(f"""
CREATE OR REPLACE TEMP VIEW observation_values AS
{_build_observation_values_select_sql()}
""")

con.execute(f"""
CREATE OR REPLACE TEMP VIEW parameter_definitions AS
{_build_parameter_definitions_select_sql()}
""")

In [210]:
# successful runs
con.sql("SELECT * FROM successful_runs").df()

,run_id,dataset_id
0,20260803T025211Z,OMNI_HRO2_1MIN
1,20260803T025255Z,OMNI_HRO2_1MIN


In [212]:
# chunks belonging to successful runs
con.sql("""
    SELECT run_id, filename, len(data) AS observation_count
    FROM successful_chunks
""").df()

,run_id,filename,observation_count
0,20260803T025211Z,temp\omni-ingest\run_id=20260803T025211Z\chunk_20260707T000000Z__20260708T004400Z.json,1484
1,20260803T025255Z,temp\omni-ingest\run_id=20260803T025255Z\chunk_20260706T000000Z__20260707T000000Z.json,1440


In [215]:
con.sql("""
    SELECT *
    FROM observation_rows
    ORDER BY run_id, filename, source_row_number
    LIMIT 3
""").df()

,dataset_id,run_id,filename,source_row_number,row_array
0,OMNI_HRO2_1MIN,20260803T025211Z,temp\omni-ingest\run_id=20260803T025211Z\chunk_20260707T000000Z__20260708T004400Z.json,1,"[""2026-07-07T00:00:00.000Z"", 52, 52, 1, 1, 100, 1960, 0, 0.0, 16, 7.26, -5.59, 4.62, 0.2, 4.6, 0.44, 0.0, 0.0, 419.1, -416.8, -39.8, -19.0, 1.14, 40271, 9.999, 0.4, -0.18, 0.15, 3.1, 2.9, 252.82, -72.13, 10.51, 18.56, -0.08, 0.91, 99999, 99999, 99999, 99999, 99999, 99999, 99999]"
1,OMNI_HRO2_1MIN,20260803T025211Z,temp\omni-ingest\run_id=20260803T025211Z\chunk_20260707T000000Z__20260708T004400Z.json,2,"[""2026-07-07T00:01:00.000Z"", 52, 99, 1, 999, 100, 2005, 0, 0.0, 15, 7.26, -5.64, 4.56, 0.16, 4.55, 0.4, 0.0, 0.0, 99999.9, 99999.9, 99999.9, 99999.9, 999.99, 9999999.0, 9.999, 99.99, 999.99, 999.99, 999.9, 99.9, 9999.99, 9999.99, 9999.99, 18.55, -0.26, 0.93, 99999, 99999, 99999, 99999, 99999, 99999, 99999]"
2,OMNI_HRO2_1MIN,20260803T025211Z,temp\omni-ingest\run_id=20260803T025211Z\chunk_20260707T000000Z__20260708T004400Z.json,3,"[""2026-07-07T00:02:00.000Z"", 52, 99, 1, 999, 100, 2051, 0, 0.0, 13, 7.23, -5.75, 4.33, 0.07, 4.32, 0.29, 0.0, 0.0, 99999.9, 99999.9, 99999.9, 99999.9, 999.99, 9999999.0, 9.999, 99.99, 999.99, 999.99, 999.9, 99.9, 9999.99, 9999.99, 9999.99, 18.5, -0.45, 0.96, 99999, 99999, 99999, 99999, 99999, 99999, 99999]"


In [216]:
con.sql("""
    SELECT *
    FROM observation_values
    ORDER BY run_id, filename, source_row_number, parameter_index
    LIMIT 5
""").df()

,dataset_id,run_id,filename,source_row_number,observation_time_utc,parameter_index,raw_value
0,OMNI_HRO2_1MIN,20260803T025211Z,temp\omni-ingest\run_id=20260803T025211Z\chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07,1,52.0
1,OMNI_HRO2_1MIN,20260803T025211Z,temp\omni-ingest\run_id=20260803T025211Z\chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07,2,52.0
2,OMNI_HRO2_1MIN,20260803T025211Z,temp\omni-ingest\run_id=20260803T025211Z\chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07,3,1.0
3,OMNI_HRO2_1MIN,20260803T025211Z,temp\omni-ingest\run_id=20260803T025211Z\chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07,4,1.0
4,OMNI_HRO2_1MIN,20260803T025211Z,temp\omni-ingest\run_id=20260803T025211Z\chunk_20260707T000000Z__20260708T004400Z.json,1,2026-07-07,5,100.0


In [217]:
con.sql("""
    SELECT *
    FROM parameter_definitions
    ORDER BY run_id, filename, parameter_index
    LIMIT 5
""").df()

,dataset_id,run_id,filename,parameter_index,parameter_name,source_fill_value,units,parameter_type
0,OMNI_HRO2_1MIN,20260803T025211Z,temp\omni-ingest\run_id=20260803T025211Z\chunk_20260707T000000Z__20260708T004400Z.json,1,IMF,99.0,None,integer
1,OMNI_HRO2_1MIN,20260803T025211Z,temp\omni-ingest\run_id=20260803T025211Z\chunk_20260707T000000Z__20260708T004400Z.json,2,PLS,99.0,None,integer
2,OMNI_HRO2_1MIN,20260803T025211Z,temp\omni-ingest\run_id=20260803T025211Z\chunk_20260707T000000Z__20260708T004400Z.json,3,IMF_PTS,999.0,None,integer
3,OMNI_HRO2_1MIN,20260803T025211Z,temp\omni-ingest\run_id=20260803T025211Z\chunk_20260707T000000Z__20260708T004400Z.json,4,PLS_PTS,999.0,None,integer
4,OMNI_HRO2_1MIN,20260803T025211Z,temp\omni-ingest\run_id=20260803T025211Z\chunk_20260707T000000Z__20260708T004400Z.json,5,percent_interp,999.0,None,integer


In [218]:
relations = [
    "successful_runs",
    "successful_chunks",
    "observation_rows",
    "observation_values",
    "parameter_definitions",
]

for relation in relations:
    count = con.sql(
        f"SELECT COUNT(*) FROM {relation}"
    ).fetchone()[0]
    print(f"{relation}: {count:,}")

successful_runs: 2
successful_chunks: 2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

observation_rows: 2,924


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

observation_values: 122,808
parameter_definitions: 84


In [219]:
long_observations = con.sql(
    build_long_observation_select_sql(
        dirs.manifest_run_dirs,
        dirs.json_glob_run_dirs,
    )
).df()

In [ ]:
long_observations.\
        query(f'run_id == "20260803T025211Z" & source_row_number == 1').\
        drop(columns=['chunk_file', 'dataset_id', 'run_id', 'source_row_number'])

,observation_time_utc,parameter_name,raw_value,source_fill_value,units,parameter_type,is_source_fill
0,2026-07-07,IMF,52.000,99.000,None,integer,False
1,2026-07-07,PLS,52.000,99.000,None,integer,False
2,2026-07-07,IMF_PTS,1.000,999.000,None,integer,False
3,2026-07-07,PLS_PTS,1.000,999.000,None,integer,False
4,2026-07-07,percent_interp,100.000,999.000,None,integer,False
5,2026-07-07,Timeshift,1960.000,999999.000,seconds,integer,False
6,2026-07-07,RMS_Timeshift,0.000,999999.000,seconds,integer,False
7,2026-07-07,RMS_phase,0.000,99.990,nT,double,False
8,2026-07-07,Time_btwn_obs,16.000,999999.000,seconds,integer,False
9,2026-07-07,F,7.260,9999.990,nT,double,False


In [235]:
(
    long_observations['is_source_fill'] ==\
        (long_observations['raw_value'] == long_observations['source_fill_value'])
).all()



True